# Training Objective Diagnostics for Rectified Flow

## Goal

This notebook checks how well the model fits its **flow-matching training objective**. It does not evaluate whether the transformation is semantically meaningful.

For a randomly paired $(x_a, x_b)$ and $t \sim U[0,1]$, the training objective is:

$$z_t=(1-t)x_a+tx_b, \qquad u=x_b-x_a, \qquad \mathcal{L}=\mathbb{E}\left[\|v_\theta(z_t,t)-u\|^2\right].$$

The notebook measures the MSE on FashionMNIST validation pairs, compares it with the trivial zero-velocity prediction, and breaks the error down by time interval. It also visualizes the interpolated input, target velocity, predicted velocity, and residual.

> **Note:** The training loop uses this split for validation and best-checkpoint selection, so these measurements are not an independent final test. Random image pairs and time samples also introduce some variation.

## Prerequisite: Colab Setup

In Colab, run `00_colab_setup.ipynb` first and make sure the repository contains the current API files. By default, this notebook loads the best checkpoint from `assets/fmnist_d2d_runs/`. If your checkpoints are stored on Google Drive, update `CHECKPOINT_DIR` below.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

repo_root = Path.cwd()
if not (repo_root / 'api' / 'flow_model_api.py').is_file():
    repo_root = Path('/content/minRF_interpolation')
    assert (repo_root / 'api' / 'flow_model_api.py').is_file(), 'Repository not found; run 00_colab_setup.ipynb first.'
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from api.flow_model_api import (
    DataToDataRF,
    DatasetSpec,
    ModelSpec,
    build_dataset_loaders,
    build_flow_model,
    load_checkpoint,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHECKPOINT_DIR = repo_root / 'assets' / 'fmnist_d2d_runs'
MODEL_NAME = 'fmnist_d2d_training'
BEST_CHECKPOINT = CHECKPOINT_DIR / f'{MODEL_NAME}_checkpoint_best.pt'
LATEST_CHECKPOINT = CHECKPOINT_DIR / f'{MODEL_NAME}_checkpoint_latest.pt'

MODEL_PROFILE = 'medium'
SOURCE_CLASS = 7
TARGET_CLASS = 9
BATCH_SIZE = 128
EVAL_BATCHES = 4  # 4 x 128 = up to 512 validation pairs
TIMES_PER_PAIR = 3
TIME_BINS = 5
USE_CUDA_AMP = DEVICE.type == 'cuda'

print(f'Device: {DEVICE}' + (f' ({torch.cuda.get_device_name(DEVICE)})' if DEVICE.type == 'cuda' else ''))
print(f'Best checkpoint: {BEST_CHECKPOINT}')
print(f'Checkpoint exists: {BEST_CHECKPOINT.is_file()}')

Device: cuda (NVIDIA A100-SXM4-80GB)
Best checkpoint: /content/minRF_interpolation/assets/fmnist_d2d_runs/fmnist_d2d_training_checkpoint_best.pt
Checkpoint exists: False


In [2]:
if not BEST_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f'Best checkpoint not found: {BEST_CHECKPOINT}. Run notebook 02 first or update CHECKPOINT_DIR.'
    )

model_spec = ModelSpec(
    model_profile=MODEL_PROFILE,
    channels=1,
    image_size=32,
    num_classes=1,
    lr=2e-4,
    pairing_mode='random',
)
model, flow_model, _ = build_flow_model(model_spec)
model = model.to(DEVICE)
best_payload = load_checkpoint(model, None, BEST_CHECKPOINT, device=DEVICE)
model.eval()

dataset_spec = DatasetSpec(
    name='fashionmnist',
    root=str(repo_root / 'data'),
    source_class=SOURCE_CLASS,
    target_class=TARGET_CLASS,
    batch_size=BATCH_SIZE,
    test_batch_size=BATCH_SIZE,
)
loaders = build_dataset_loaders(dataset_spec, download=True)
val_loader = loaders['test_loader']
flow_model = DataToDataRF(model, pairing_mode='random')

checkpoint_config = best_payload.get('config', {})
print(f"Loaded checkpoint: {BEST_CHECKPOINT.name}")
print(f"Checkpoint model profile: {checkpoint_config.get('model_profile', MODEL_PROFILE)}")
print(f"Available validation pairs: {len(loaders['test_dataset'])}")

FileNotFoundError: Best checkpoint not found: /content/minRF_interpolation/assets/fmnist_d2d_runs/fmnist_d2d_training_checkpoint_best.pt. Run notebook 02 first or update CHECKPOINT_DIR.

## 1. Training History

The `best` checkpoint contains the history up to the epoch when it was saved. If available, use the longer history from `latest`.

In [ ]:
history_payload = best_payload
if LATEST_CHECKPOINT.is_file():
    history_payload = torch.load(LATEST_CHECKPOINT, map_location='cpu', weights_only=False)

train_history = history_payload.get('train_history', [])
val_history = history_payload.get('val_history', [])
if train_history and val_history:
    epochs = np.arange(1, min(len(train_history), len(val_history)) + 1)
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(epochs, train_history[:len(epochs)], label='Training MSE')
    ax.plot(epochs, val_history[:len(epochs)], label='Validation MSE')
    best_epoch = int(np.argmin(val_history)) + 1
    ax.scatter([best_epoch], [val_history[best_epoch - 1]], color='tab:red', zorder=3,
               label=f'Best saved validation loss (epoch {best_epoch})')
    ax.set(title='Flow-Matching Loss During Training', xlabel='Epoch', ylabel='MSE per pixel')
    ax.grid(alpha=0.25)
    ax.legend()
    plt.show()
    print(f'Epochs in checkpoint history: {len(epochs)}')
else:
    print('No loss history found in the checkpoint; the objective evaluation below can still run.')

## 2. Flow-Matching Objective on Validation Pairs

For each image pair, sample several independent time points $t$. The **zero-velocity baseline MSE** always predicts a velocity of zero and shows whether the model matches the target velocity better than this trivial reference. The metric averages squared error over all pixels and channels.

In [ ]:
objective_errors = []
zero_baseline_errors = []
cosine_scores = []
time_error_bins = [[] for _ in range(TIME_BINS)]
viz_source = None
viz_target = None

with torch.inference_mode():
    for batch_index, (x_a, x_b) in enumerate(val_loader):
        if batch_index >= EVAL_BATCHES:
            break
        x_a = x_a.to(DEVICE)
        x_b = x_b.to(DEVICE)
        if viz_source is None:
            viz_source = x_a[:4].clone()
            viz_target = x_b[:4].clone()

        target_velocity = x_b - x_a
        zero_mse = target_velocity.float().square().flatten(1).mean(dim=1)
        for _ in range(TIMES_PER_PAIR):
            t = torch.rand(x_a.shape[0], device=DEVICE)
            t_view = t.view(-1, 1, 1, 1)
            z_t = (1.0 - t_view) * x_a + t_view * x_b
            cond = torch.zeros(x_a.shape[0], dtype=torch.long, device=DEVICE)
            with torch.autocast('cuda', dtype=torch.float16, enabled=USE_CUDA_AMP):
                pred_velocity = model(z_t, t, cond)

            pred_float = pred_velocity.float()
            target_float = target_velocity.float()
            sample_mse = (pred_float - target_float).square().flatten(1).mean(dim=1)
            cosine = F.cosine_similarity(
                pred_float.flatten(1), target_float.flatten(1), dim=1, eps=1e-8
            )
            objective_errors.append(sample_mse.cpu())
            zero_baseline_errors.append(zero_mse.cpu())
            cosine_scores.append(cosine.cpu())

            bin_ids = torch.clamp((t * TIME_BINS).long(), max=TIME_BINS - 1)
            for bin_id in range(TIME_BINS):
                mask = bin_ids == bin_id
                if mask.any():
                    time_error_bins[bin_id].append(sample_mse[mask].cpu())

objective_errors = torch.cat(objective_errors).numpy()
zero_baseline_errors = torch.cat(zero_baseline_errors).numpy()
cosine_scores = torch.cat(cosine_scores).numpy()
objective_mse = float(objective_errors.mean())
zero_mse = float(zero_baseline_errors.mean())
relative_improvement = 1.0 - objective_mse / zero_mse if zero_mse > 0 else float('nan')

print(f'Pairs: up to {EVAL_BATCHES * BATCH_SIZE}; random t samples per pair: {TIMES_PER_PAIR}')
print(f'Flow-matching MSE:             {objective_mse:.6f}')
print(f'Zero-velocity baseline MSE:    {zero_mse:.6f}')
print(f'Relative MSE reduction vs zero: {100 * relative_improvement:.1f}%')
print(f'Mean velocity cosine similarity: {cosine_scores.mean():.4f}')
print(f'90th percentile pairwise MSE: {np.quantile(objective_errors, 0.90):.6f}')

## 3. Error Distribution and Dependence on $t$

A mean alone can hide difficult pairs. The histogram shows the distribution of pairwise errors; the second plot shows whether the objective is fitted noticeably worse in particular time intervals. Error bars indicate the standard error of the mean within each interval.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(objective_errors, bins=35, alpha=0.8, label='Model')
axes[0].axvline(objective_mse, color='tab:blue', linestyle='--', label=f'Model mean {objective_mse:.4f}')
axes[0].axvline(zero_mse, color='tab:red', linestyle=':', label=f'Zero-baseline mean {zero_mse:.4f}')
axes[0].set(title='Flow-matching MSE per pair and t', xlabel='MSE', ylabel='Count')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.2)

bin_means = []
bin_sems = []
bin_labels = []
for bin_index, values in enumerate(time_error_bins):
    if values:
        values = torch.cat(values).numpy()
        bin_means.append(float(values.mean()))
        bin_sems.append(float(values.std(ddof=1) / np.sqrt(len(values))) if len(values) > 1 else 0.0)
        bin_labels.append(f'{bin_index / TIME_BINS:.1f}–{(bin_index + 1) / TIME_BINS:.1f}')
axes[1].errorbar(bin_labels, bin_means, yerr=bin_sems, marker='o', capsize=4, label='Model MSE')
axes[1].axhline(zero_mse, color='tab:red', linestyle=':', label='Zero-baseline MSE')
axes[1].set(title='Objective Error by Time Interval', xlabel='Time interval t', ylabel='MSE')
axes[1].grid(alpha=0.25)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## 4. Target and Predicted Velocity in Image Space

Each row shows a validation pair at $t=0.5$. All velocity images share the same color scale. The residual is $v_\theta(z_t,t)-(x_b-x_a)$; it shows where the prediction differs from the training target. These examples illustrate the objective and are not selected for especially good results.

In [ ]:
n_viz = min(4, len(viz_source))
t_value = 0.5
with torch.inference_mode():
    source = viz_source[:n_viz].to(DEVICE)
    target = viz_target[:n_viz].to(DEVICE)
    t = torch.full((n_viz,), t_value, device=DEVICE)
    z_t = (1.0 - t.view(-1, 1, 1, 1)) * source + t.view(-1, 1, 1, 1) * target
    cond = torch.zeros(n_viz, dtype=torch.long, device=DEVICE)
    with torch.autocast('cuda', dtype=torch.float16, enabled=USE_CUDA_AMP):
        pred = model(z_t, t, cond).float()
    target_velocity = (target - source).float()
    residual = pred - target_velocity

shared_limit = max(
    float(target_velocity.abs().max()),
    float(pred.abs().max()),
    float(residual.abs().max()),
    1e-6,
)
fig, axes = plt.subplots(n_viz, 6, figsize=(15, 2.8 * n_viz), squeeze=False)
for row in range(n_viz):
    panels = [
        ((source[row, 0].cpu() + 1) / 2, 'Source xₐ', 'gray', None),
        ((z_t[row, 0].cpu() + 1) / 2, 'Interpolant zₜ, t=0.5', 'gray', None),
        ((target[row, 0].cpu() + 1) / 2, 'Target xᵦ', 'gray', None),
        (pred[row, 0].cpu(), 'Prediction vθ', 'coolwarm', shared_limit),
        (target_velocity[row, 0].cpu(), 'Target velocity xᵦ−xₐ', 'coolwarm', shared_limit),
        (residual[row, 0].cpu(), 'Residual (prediction − target)', 'coolwarm', shared_limit),
    ]
    for col, (image, title, cmap, limit) in enumerate(panels):
        axes[row, col].imshow(image, cmap=cmap, vmin=-limit if limit else None, vmax=limit)
        axes[row, col].set_title(title, fontsize=9)
        axes[row, col].axis('off')
fig.suptitle(f'Flow-matching examples on validation pairs (t={t_value})', y=1.01)
plt.tight_layout()
plt.show()
print(f'Examples shown: {n_viz}; shared velocity color range: ±{shared_limit:.3f}')

## Interpretation and Limitations

- **Flow-matching MSE:** directly measures the squared velocity error optimized during training on the sampled validation pairs and time points.
- **Zero-velocity baseline:** a useful reference; a lower model MSE means the model approximates the random target velocity better than predicting zero.
- **MSE by $t$:** can indicate time intervals where the regression target may be harder to fit. Differences are not automatically statistically meaningful because pair and time samples are random.
- **Velocity images:** show whether predicted changes are spatially aligned with the target changes.

Note: These checks do **not** show that integrated trajectories are realistic, semantically meaningful, or better than other methods! 